# exp015_public_pf_beam_scale_selector_features train

Notebook-first CV for conservative postprocess and small model-diversity candidates on top of the exp012 LightGBM residual anchor.


## Contents

1. Setup and configuration
2. Metric, data, and variant helpers
3. CV scoring and group-summary helpers
4. Fold-safe anchor/diversity CV run
5. Metrics and artifacts


## 1. Setup and configuration


In [ ]:
from __future__ import annotations

import json
import os
from datetime import UTC, datetime
from math import sqrt
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold

from baseline import (
    HORIZONTAL_SUFFIX,
    active_feature_columns,
    build_drift_feature_frame,
    config_get,
    drift_strategy,
    fit_drift_model_from_files,
    optional_positive_int,
    distance_bucket_alphas,
    predict_drift,
    read_typewell_for_horizontal_path,
    smooth_prediction,
    primary_strategy,
    well_id_from_path,
)
from settings import EXPERIMENT_NAME, ExperimentPaths, deep_merge, load_config

DEBUG = os.environ.get("EXPERIMENT_DEBUG", "0") == "1"
MAX_WELLS_ENV = os.environ.get("EXPERIMENT_MAX_WELLS")
MAX_WELLS = int(MAX_WELLS_ENV) if MAX_WELLS_ENV else None
VARIANT_LIMIT_ENV = os.environ.get("EXPERIMENT_VARIANT_LIMIT")
VARIANT_LIMIT = int(VARIANT_LIMIT_ENV) if VARIANT_LIMIT_ENV else None

paths = ExperimentPaths()
paths.require_kaggle_runtime()
paths.ensure_output_dirs()
config = load_config()
primary = primary_strategy(config)
drift_name = drift_strategy(config)

print("Experiment:", EXPERIMENT_NAME)
print("Root:", paths.root)
print("Train data:", paths.train_data_dir)
print("Output root:", paths.output_root)
print("Artifacts:", paths.artifacts_dir)
print("Primary strategy:", primary)
print("Drift strategy:", drift_name)
print("Base feature columns:", len(active_feature_columns(config)))
print("Base estimator:", config_get(config, "model.drift_model.estimator", "HistGradientBoostingRegressor"))
print("Postprocess selection variant:", config_get(config, "postprocess.selection_variant", ""))
print("Selected postprocess:", config_get(config, "postprocess.selected_method", "raw"))
print("Debug:", DEBUG, "Max wells:", MAX_WELLS, "Variant limit:", VARIANT_LIMIT)


## 2. Metric, Data, And Variant Helpers


In [ ]:
def finite_float(value: float | None, digits: int = 6) -> float | None:
    if value is None or not np.isfinite(value):
        return None
    return round(float(value), digits)


def rmse_from_sums(sse: float, count: int) -> float | None:
    if count <= 0:
        return None
    return sqrt(sse / count)


def rmse_from_arrays(y_true: np.ndarray, y_pred: np.ndarray) -> float | None:
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    if not valid.any():
        return None
    residual = y_pred[valid] - y_true[valid]
    return float(np.sqrt(np.mean(residual * residual)))


def train_files(paths: ExperimentPaths, cfg: dict[str, Any], debug: bool, max_wells: int | None) -> list[Path]:
    files = sorted(paths.train_data_dir.glob(f"*{HORIZONTAL_SUFFIX}"))
    if not files:
        raise FileNotFoundError(f"no train horizontal well CSVs found in {paths.train_data_dir}")
    if debug:
        limit = max_wells
        if limit is None:
            limit = int(cfg.get("runtime", {}).get("debug_n_wells", 30))
        files = files[:limit]
    elif max_wells is not None:
        files = files[:max_wells]
    return files


def build_fold_map(files: list[Path], n_folds: int) -> tuple[dict[str, int], int]:
    well_ids = [well_id_from_path(path) for path in files]
    n_splits = min(max(1, n_folds), len(well_ids))
    if n_splits < 2:
        raise ValueError("drift residual CV requires at least two wells")

    fold_map: dict[str, int] = {}
    x = np.arange(len(well_ids)).reshape(-1, 1)
    groups = np.asarray(well_ids)
    splitter = GroupKFold(n_splits=n_splits)
    for fold, (_, valid_idx) in enumerate(splitter.split(x, groups=groups)):
        for index in valid_idx:
            fold_map[well_ids[int(index)]] = fold
    return fold_map, n_splits


def configured_row_cap(cfg: dict[str, Any], key: str) -> int | None:
    return optional_positive_int(config_get(cfg, key, None))


def ablation_variants(cfg: dict[str, Any]) -> list[dict[str, Any]]:
    raw_variants = config_get(cfg, "ablation.variants", [])
    if not isinstance(raw_variants, list) or not raw_variants:
        raw_variants = [{"name": "control", "variable": "control", "description": "base config", "overrides": {}}]

    variants: list[dict[str, Any]] = []
    for index, variant in enumerate(raw_variants):
        if not isinstance(variant, dict):
            raise ValueError(f"ablation variant #{index} must be a mapping")
        if variant.get("enabled", True) is False:
            continue
        name = str(variant.get("name") or f"variant_{index}")
        overrides = variant.get("overrides") or {}
        if not isinstance(overrides, dict):
            raise ValueError(f"ablation variant {name} overrides must be a mapping")
        variants.append(
            {
                "name": name,
                "variable": str(variant.get("variable") or "unknown"),
                "description": str(variant.get("description") or ""),
                "overrides": overrides,
            }
        )

    if VARIANT_LIMIT is not None:
        variants = variants[:VARIANT_LIMIT]
    if not variants:
        raise ValueError("no enabled ablation variants")
    return variants


def config_for_variant(base_cfg: dict[str, Any], variant: dict[str, Any]) -> dict[str, Any]:
    variant_cfg = deep_merge(base_cfg, variant["overrides"])
    return variant_cfg


def delta_vs_baseline(value: float | None, baseline: float | None) -> float | None:
    if value is None or baseline is None:
        return None
    return finite_float(float(value) - float(baseline))


## 3. CV Scoring And Group-Summary Helpers


In [ ]:
def add_score(
    stats: dict[str, Any],
    strategy: str,
    fold: int,
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> None:
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    if not valid.any():
        return
    residual = y_pred[valid] - y_true[valid]
    sse = float(np.sum(residual * residual))
    count = int(valid.sum())
    stats[strategy]["fold_sse"][fold] += sse
    stats[strategy]["fold_n"][fold] += count
    stats[strategy]["total_sse"] += sse
    stats[strategy]["total_n"] += count
    stats[strategy]["well_rmse"].append(float(np.sqrt(sse / count)))


def summarize_strategy(strategy_stats: dict[str, Any]) -> dict[str, Any]:
    fold_rmse = [
        finite_float(rmse_from_sums(sse, int(count)))
        for sse, count in zip(strategy_stats["fold_sse"], strategy_stats["fold_n"], strict=True)
    ]
    valid_fold_rmse = [value for value in fold_rmse if value is not None]
    well_rmse = np.asarray(strategy_stats["well_rmse"], dtype=float)
    return {
        "oof_rmse": finite_float(
            rmse_from_sums(strategy_stats["total_sse"], int(strategy_stats["total_n"]))
        ),
        "mean_fold_rmse": finite_float(
            float(np.mean(valid_fold_rmse)) if valid_fold_rmse else None
        ),
        "fold_rmse": fold_rmse,
        "rows": int(strategy_stats["total_n"]),
        "per_well_rmse_mean": finite_float(float(np.mean(well_rmse)) if well_rmse.size else None),
        "per_well_rmse_median": finite_float(
            float(np.median(well_rmse)) if well_rmse.size else None
        ),
    }


def finite_missing_rate(values: np.ndarray) -> float | None:
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        return None
    return finite_float(float(np.mean(~np.isfinite(values))))


def finite_abs_median(values: pd.Series) -> float | None:
    finite = values.to_numpy(dtype=float)
    finite = finite[np.isfinite(finite)]
    if finite.size == 0:
        return None
    return finite_float(float(np.median(np.abs(finite))))


def model_group_summary(well_df: pd.DataFrame, model_strategy: str) -> pd.DataFrame:
    rmse_col = f"{model_strategy}_rmse"
    if well_df.empty or rmse_col not in well_df.columns:
        return pd.DataFrame()

    base = well_df.sort_values(["variant", "well_id"]).drop_duplicates("well_id").set_index("well_id")
    control_all = well_df[well_df["variant"] == "control_hgb_all"].set_index("well_id")
    control_no_gr = well_df[well_df["variant"] == "control_hgb_no_gr"].set_index("well_id")
    base["control_hgb_all_rmse"] = control_all[rmse_col] if rmse_col in control_all else np.nan
    base["control_hgb_no_gr_rmse"] = control_no_gr[rmse_col] if rmse_col in control_no_gr else np.nan

    gr_weak = (base["prefix_gr_missing_rate"] >= 0.35) | (base["eval_gr_missing_rate"] >= 0.40)
    gr_strong = (base["prefix_gr_missing_rate"] < 0.20) & (base["eval_gr_missing_rate"] < 0.25)
    no_gr_better = base["control_hgb_no_gr_rmse"] + 0.10 < base["control_hgb_all_rmse"]
    all_gr_better = base["control_hgb_all_rmse"] + 0.10 < base["control_hgb_no_gr_rmse"]
    exp002_hard = base["control_hgb_all_rmse"] > base["last_anchor_rmse"] + 0.10

    group_masks = {
        "all": pd.Series(True, index=base.index),
        "hard_no_gr_candidate": no_gr_better & (gr_weak | exp002_hard),
        "public_like_keep_all_gr": all_gr_better & gr_strong,
        "high_gr_missing": base["eval_gr_missing_rate"] >= 0.40,
        "long_eval": base["eval_row_count"] >= 5700,
        "steep_trajectory": base["trajectory_abs_dz_dmd"] >= 0.04,
    }

    rows: list[dict[str, Any]] = []
    for variant, variant_df in well_df.groupby("variant"):
        variant_by_well = variant_df.set_index("well_id")
        for group_name, mask in group_masks.items():
            group_ids = mask.index[mask.fillna(False)]
            group_df = variant_by_well.loc[variant_by_well.index.intersection(group_ids)]
            if group_df.empty:
                continue
            rows.append(
                {
                    "variant": variant,
                    "group": group_name,
                    "wells": int(group_df.shape[0]),
                    "mean_model_rmse": finite_float(float(group_df[rmse_col].mean())),
                    "median_model_rmse": finite_float(float(group_df[rmse_col].median())),
                    "mean_last_anchor_rmse": finite_float(float(group_df["last_anchor_rmse"].mean())),
                    "mean_prefix_gr_missing_rate": finite_float(float(group_df["prefix_gr_missing_rate"].mean())),
                    "mean_eval_gr_missing_rate": finite_float(float(group_df["eval_gr_missing_rate"].mean())),
                    "mean_eval_row_count": finite_float(float(group_df["eval_row_count"].mean())),
                    "rmse_column": rmse_col,
                }
            )
    return pd.DataFrame(rows)


def write_csv_artifacts(
    paths: ExperimentPaths,
    ablation_records: list[dict[str, Any]],
    well_records: list[dict[str, Any]],
    fold_records: list[dict[str, Any]],
    model_records: list[dict[str, Any]],
    model_strategy: str,
) -> None:
    if ablation_records:
        pd.DataFrame(ablation_records).to_csv(
            paths.artifacts_dir / "ablation_metrics.csv", index=False
        )
    if well_records:
        well_df = pd.DataFrame(well_records)
        well_df.to_csv(paths.artifacts_dir / "well_metrics.csv", index=False)
        group_df = model_group_summary(well_df, model_strategy)
        if not group_df.empty:
            group_df.to_csv(paths.artifacts_dir / "model_group_summary.csv", index=False)
    if fold_records:
        pd.DataFrame(fold_records).to_csv(paths.artifacts_dir / "fold_metrics.csv", index=False)
    if model_records:
        pd.DataFrame(model_records).to_csv(
            paths.artifacts_dir / "fold_model_training.csv", index=False
        )



def configured_oof_variants(cfg: dict[str, Any]) -> set[str]:
    names = {
        str(config_get(cfg, "postprocess.selection_variant", "lightgbm_no_gr")),
        str(config_get(cfg, "postprocess.diversity_control_variant", "control_hgb_no_gr")),
    }
    return {name for name in names if name}


def add_oof_rows(
    oof_records: list[dict[str, Any]],
    *,
    variant: str,
    variable: str,
    fold: int,
    well_id: str,
    frame: Any,
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> None:
    for row_index, eval_step, last_anchor, truth, pred in zip(
        frame.eval_indices,
        frame.features["eval_step"].to_numpy(dtype=float),
        frame.baseline_prediction,
        y_true,
        y_pred,
        strict=True,
    ):
        oof_records.append(
            {
                "variant": variant,
                "variable": variable,
                "fold": fold,
                "well_id": well_id,
                "row_id": f"{well_id}_{int(row_index)}",
                "row_index": int(row_index),
                "eval_step": int(eval_step),
                "eval_row_count": int(frame.eval_indices.size),
                "last_anchor": float(last_anchor),
                "y_true": float(truth),
                "y_pred": float(pred),
                "residual_pred": float(pred - last_anchor),
                "residual_true": float(truth - last_anchor),
            }
        )


def postprocess_oof_arrays(
    method: str,
    params: dict[str, Any],
    raw_prediction: np.ndarray,
    last_anchor: np.ndarray,
    eval_step: np.ndarray,
) -> np.ndarray:
    raw_prediction = np.asarray(raw_prediction, dtype=float)
    last_anchor = np.asarray(last_anchor, dtype=float)
    eval_step = np.asarray(eval_step, dtype=float)
    residual = raw_prediction - last_anchor

    if method in {"raw", "raw_lightgbm_no_gr"}:
        return raw_prediction.copy()
    if method == "sg_smooth":
        # Smoothing must stay within each well; caller applies this through groupby.
        raise ValueError("sg_smooth is evaluated by postprocess_sg_by_well")
    if method == "global_residual_shrink":
        return last_anchor + float(params["alpha"]) * residual
    if method == "near_anchor_damping":
        near_rows = max(1.0, float(params["near_rows"]))
        near_alpha = float(params["near_alpha"])
        far_alpha = float(params.get("far_alpha", 1.0))
        progress = np.clip(eval_step / near_rows, 0.0, 1.0)
        alpha = near_alpha + (far_alpha - near_alpha) * progress
        return last_anchor + alpha * residual
    if method == "distance_bucket_shrink":
        alpha = distance_bucket_alphas(eval_step, params.get("buckets", []))
        return last_anchor + alpha * residual
    raise ValueError(f"unsupported OOF postprocess method: {method}")


def postprocess_sg_by_well(oof_df: pd.DataFrame, params: dict[str, Any]) -> np.ndarray:
    output = np.full(oof_df.shape[0], np.nan, dtype=float)
    for _, group in oof_df.groupby("well_id", sort=False):
        order = group.sort_values("eval_step").index
        raw_values = group.loc[order, "y_pred"].to_numpy(dtype=float)
        smoothed = smooth_prediction(
            raw_values,
            window=int(params.get("window", 31)),
            polyorder=int(params.get("polyorder", 2)),
        )
        blend = float(params.get("blend", 1.0))
        output[oof_df.index.get_indexer(order)] = raw_values * (1.0 - blend) + smoothed * blend
    return output


def postprocess_rmse(oof_df: pd.DataFrame, y_pred: np.ndarray) -> float | None:
    return finite_float(rmse_from_arrays(oof_df["y_true"].to_numpy(dtype=float), y_pred))


def postprocess_record(
    name: str,
    method: str,
    params: dict[str, Any],
    oof_df: pd.DataFrame,
    y_pred: np.ndarray,
    *,
    source: str,
) -> dict[str, Any]:
    raw_rmse = postprocess_rmse(oof_df, oof_df["y_pred"].to_numpy(dtype=float))
    rmse = postprocess_rmse(oof_df, y_pred)
    return {
        "candidate": name,
        "method": method,
        "source": source,
        "rmse": rmse,
        "delta_vs_raw": delta_vs_baseline(rmse, raw_rmse),
        "rows": int(oof_df.shape[0]),
        "params_json": json.dumps(params, sort_keys=True),
    }


def fitted_distance_bucket_params(oof_df: pd.DataFrame, cfg: dict[str, Any]) -> dict[str, Any]:
    buckets = config_get(cfg, "postprocess.distance_buckets", [])
    if not isinstance(buckets, list) or not buckets:
        return {"buckets": []}
    clip_min = float(config_get(cfg, "postprocess.alpha_clip.min", 0.20))
    clip_max = float(config_get(cfg, "postprocess.alpha_clip.max", 1.15))
    rows: list[dict[str, Any]] = []
    previous_max = -np.inf
    for bucket in buckets:
        max_step = float(bucket["max_step"])
        mask = (oof_df["eval_step"] > previous_max) & (oof_df["eval_step"] <= max_step)
        part = oof_df.loc[mask]
        residual_pred = part["residual_pred"].to_numpy(dtype=float)
        residual_true = part["residual_true"].to_numpy(dtype=float)
        denom = float(np.dot(residual_pred, residual_pred))
        alpha = 1.0 if denom <= 0.0 else float(np.dot(residual_pred, residual_true) / denom)
        alpha = float(np.clip(alpha, clip_min, clip_max))
        rows.append(
            {
                "name": str(bucket.get("name", f"step_le_{int(max_step)}")),
                "max_step": max_step,
                "alpha": alpha,
            }
        )
        previous_max = max_step
    return {"buckets": rows}


def evaluate_postprocess_candidates(
    oof_records: list[dict[str, Any]],
    cfg: dict[str, Any],
    paths: ExperimentPaths,
) -> tuple[list[dict[str, Any]], dict[str, Any] | None]:
    if not oof_records:
        return [], None

    oof_all = pd.DataFrame(oof_records)
    oof_all.to_csv(paths.artifacts_dir / "row_oof_predictions.csv", index=False)

    selection_variant = str(config_get(cfg, "postprocess.selection_variant", "lightgbm_no_gr"))
    anchor_df = oof_all[oof_all["variant"] == selection_variant].copy()
    if anchor_df.empty:
        return [], None
    anchor_df = anchor_df.sort_values(["well_id", "row_index"]).reset_index(drop=True)

    records: list[dict[str, Any]] = []
    raw_pred = anchor_df["y_pred"].to_numpy(dtype=float)
    last_anchor = anchor_df["last_anchor"].to_numpy(dtype=float)
    eval_step = anchor_df["eval_step"].to_numpy(dtype=float)
    records.append(postprocess_record("raw_lightgbm_no_gr", "raw", {}, anchor_df, raw_pred, source=selection_variant))

    candidates = config_get(cfg, "postprocess.candidates", {})
    if not isinstance(candidates, dict):
        candidates = {}

    for alpha in candidates.get("global_residual_shrink_alphas", []):
        params = {"alpha": float(alpha)}
        pred = postprocess_oof_arrays("global_residual_shrink", params, raw_pred, last_anchor, eval_step)
        records.append(
            postprocess_record(
                f"global_residual_shrink_{float(alpha):.2f}",
                "global_residual_shrink",
                params,
                anchor_df,
                pred,
                source=selection_variant,
            )
        )

    for params in candidates.get("near_anchor_damping", []):
        if not isinstance(params, dict):
            continue
        pred = postprocess_oof_arrays("near_anchor_damping", params, raw_pred, last_anchor, eval_step)
        records.append(
            postprocess_record(
                f"near_anchor_damping_{int(params['near_rows'])}_{float(params['near_alpha']):.2f}",
                "near_anchor_damping",
                params,
                anchor_df,
                pred,
                source=selection_variant,
            )
        )

    for params in candidates.get("sg_smooth", []):
        if not isinstance(params, dict):
            continue
        pred = postprocess_sg_by_well(anchor_df, params)
        records.append(
            postprocess_record(
                f"sg_smooth_w{int(params['window'])}_b{float(params.get('blend', 1.0)):.2f}",
                "sg_smooth",
                params,
                anchor_df,
                pred,
                source=selection_variant,
            )
        )

    bucket_params = fitted_distance_bucket_params(anchor_df, cfg)
    bucket_pred = postprocess_oof_arrays(
        "distance_bucket_shrink", bucket_params, raw_pred, last_anchor, eval_step
    )
    records.append(
        postprocess_record(
            "distance_bucket_shrink_fit",
            "distance_bucket_shrink",
            bucket_params,
            anchor_df,
            bucket_pred,
            source=selection_variant,
        )
    )

    control_variant = str(config_get(cfg, "postprocess.diversity_control_variant", "control_hgb_no_gr"))
    control_df = oof_all[oof_all["variant"] == control_variant][["row_id", "y_pred"]].rename(
        columns={"y_pred": "control_pred"}
    )
    if not control_df.empty:
        blend_df = anchor_df.merge(control_df, on="row_id", how="inner")
        if not blend_df.empty:
            lightgbm_pred = blend_df["y_pred"].to_numpy(dtype=float)
            hgb_pred = blend_df["control_pred"].to_numpy(dtype=float)
            for weight in candidates.get("blend_lightgbm_weights", []):
                weight = float(weight)
                pred = weight * lightgbm_pred + (1.0 - weight) * hgb_pred
                records.append(
                    postprocess_record(
                        f"hgb_lightgbm_blend_lgb_{weight:.2f}",
                        "hgb_lightgbm_blend",
                        {"lightgbm_weight": weight, "hgb_weight": 1.0 - weight},
                        blend_df,
                        pred,
                        source=f"{selection_variant}+{control_variant}",
                    )
                )

    metrics_df = pd.DataFrame(records).sort_values("rmse", na_position="last")
    metrics_df.to_csv(paths.artifacts_dir / "postprocess_metrics.csv", index=False)

    bucket_rows: list[dict[str, Any]] = []
    for bucket in bucket_params.get("buckets", []):
        max_step = float(bucket["max_step"])
        previous = max(
            [float(other["max_step"]) for other in bucket_params["buckets"] if float(other["max_step"]) < max_step],
            default=-np.inf,
        )
        mask = (anchor_df["eval_step"] > previous) & (anchor_df["eval_step"] <= max_step)
        part = anchor_df.loc[mask]
        if part.empty:
            continue
        raw_bucket_rmse = postprocess_rmse(part, part["y_pred"].to_numpy(dtype=float))
        fitted_pred = postprocess_oof_arrays(
            "global_residual_shrink",
            {"alpha": float(bucket["alpha"])},
            part["y_pred"].to_numpy(dtype=float),
            part["last_anchor"].to_numpy(dtype=float),
            part["eval_step"].to_numpy(dtype=float),
        )
        bucket_rows.append(
            {
                "bucket": bucket["name"],
                "max_step": max_step,
                "rows": int(part.shape[0]),
                "alpha": finite_float(float(bucket["alpha"])),
                "last_anchor_rmse": postprocess_rmse(part, part["last_anchor"].to_numpy(dtype=float)),
                "raw_rmse": raw_bucket_rmse,
                "bucket_shrink_rmse": postprocess_rmse(part, fitted_pred),
            }
        )
    if bucket_rows:
        pd.DataFrame(bucket_rows).to_csv(
            paths.artifacts_dir / "postprocess_distance_bucket_summary.csv", index=False
        )

    best = metrics_df.iloc[0].to_dict() if not metrics_df.empty else None
    if best is not None:
        selected_payload = {
            "candidate": best["candidate"],
            "method": best["method"],
            "source": best["source"],
            "rmse": finite_float(float(best["rmse"])),
            "params": json.loads(best["params_json"]),
        }
        (paths.artifacts_dir / "postprocess_selected_params.json").write_text(
            json.dumps(selected_payload, indent=2) + "\n"
        )
        best = selected_payload
    return records, best


## 4. Fold-Safe Model-Class CV Run


In [ ]:
files = train_files(paths, config, debug=DEBUG, max_wells=MAX_WELLS)
fold_map, n_splits = build_fold_map(files, int(config["validation"]["n_folds"]))
variants = ablation_variants(config)
baseline_cv = config_get(config, "ablation.baseline_cv", None)
cv_anchor_cv = config_get(config, "ablation.cv_anchor_cv", None)
target_column = config["data"]["target_column"]
seed = int(config["validation"]["seed"])

print("Anchor/diversity variants:", [variant["name"] for variant in variants])
print("Wells:", len(files), "Folds:", n_splits)

variant_summaries: dict[str, Any] = {}
ablation_records: list[dict[str, Any]] = []
well_records: list[dict[str, Any]] = []
fold_records: list[dict[str, Any]] = []
model_records: list[dict[str, Any]] = []
oof_records: list[dict[str, Any]] = []
oof_variant_names = configured_oof_variants(config)

for variant_index, variant in enumerate(variants):
    variant_name = variant["name"]
    variant_config = config_for_variant(config, variant)
    variant_primary = primary_strategy(variant_config)
    variant_drift = drift_strategy(variant_config)
    strategy_list = variant_config["model"]["strategies"]
    feature_columns = active_feature_columns(variant_config)
    max_rows_per_fold = configured_row_cap(variant_config, "model.training.max_train_rows_per_fold")
    max_rows_per_well = configured_row_cap(variant_config, "model.training.max_train_rows_per_well")
    max_rows_final = configured_row_cap(variant_config, "model.training.max_train_rows_final")
    estimator = str(config_get(variant_config, "model.drift_model.estimator", "HistGradientBoostingRegressor"))

    print(
        f"Variant {variant_index + 1}/{len(variants)} {variant_name}: "
        f"estimator={estimator}, "
        f"feature_set={config_get(variant_config, 'model.feature_set', 'all')}, "
        f"features={len(feature_columns)}, max_fold={max_rows_per_fold}, "
        f"max_final={max_rows_final}, max_per_well={max_rows_per_well}, "
        f"shrink={config_get(variant_config, 'model.params.residual_shrink', None)}"
    )

    stats = {
        strategy: {
            "fold_sse": np.zeros(n_splits, dtype=float),
            "fold_n": np.zeros(n_splits, dtype=np.int64),
            "total_sse": 0.0,
            "total_n": 0,
            "well_rmse": [],
        }
        for strategy in strategy_list
    }

    for fold in range(n_splits):
        train_fold_files = [path for path in files if fold_map[well_id_from_path(path)] != fold]
        valid_fold_files = [path for path in files if fold_map[well_id_from_path(path)] == fold]
        print(
            f"  Fold {fold}: fitting on {len(train_fold_files)} wells, "
            f"validating on {len(valid_fold_files)} wells"
        )
        model, n_train_rows = fit_drift_model_from_files(
            train_fold_files,
            variant_config,
            seed=seed + fold,
            max_rows_total=max_rows_per_fold,
            max_rows_per_well=max_rows_per_well,
        )
        model_records.append(
            {
                "variant": variant_name,
                "variable": variant["variable"],
                "fold": fold,
                "n_train_wells": len(train_fold_files),
                "n_valid_wells": len(valid_fold_files),
                "n_train_rows": n_train_rows,
                "max_rows_per_fold": max_rows_per_fold,
                "max_rows_final": max_rows_final,
                "max_rows_per_well": max_rows_per_well,
                "feature_set": config_get(variant_config, "model.feature_set", "all"),
                "estimator": estimator,
                "n_features": len(feature_columns),
                "residual_shrink": config_get(variant_config, "model.params.residual_shrink", None),
            }
        )

        for path in valid_fold_files:
            well_id = well_id_from_path(path)
            df = pd.read_csv(path)
            typewell_df = read_typewell_for_horizontal_path(path)
            frame = build_drift_feature_frame(
                df,
                variant_config,
                include_target=True,
                typewell_df=typewell_df,
            )
            y_true = df.loc[frame.eval_indices, target_column].to_numpy(dtype=float)
            prefix_gr = df.loc[: frame.last_known_index, "GR"].to_numpy(dtype=float)
            eval_gr = df.loc[frame.eval_indices, "GR"].to_numpy(dtype=float)
            predictions = {
                "last_anchor": frame.baseline_prediction,
                variant_drift: predict_drift(frame, model, variant_config),
            }
            record: dict[str, Any] = {
                "variant": variant_name,
                "variable": variant["variable"],
                "well_id": well_id,
                "fold": fold,
                "n_rows": int(len(df)),
                "n_known": int(frame.last_known_index + 1),
                "n_eval": int(frame.eval_indices.size),
                "last_known_index": frame.last_known_index,
                "last_known_md": frame.last_known_md,
                "last_known_tvt": frame.last_known_tvt,
                "recent_slope": frame.recent_slope,
                "prefix_gr_missing_rate": finite_missing_rate(prefix_gr),
                "eval_gr_missing_rate": finite_missing_rate(eval_gr),
                "eval_row_count": float(frame.eval_indices.size),
                "trajectory_abs_dz_dmd": finite_abs_median(frame.features["anchor_dz_dmd"]),
                "target_residual_mean": finite_float(
                    float(np.nanmean(frame.target_residual))
                    if frame.target_residual is not None and frame.target_residual.size
                    else None
                ),
            }
            for strategy in strategy_list:
                y_pred = predictions[strategy]
                add_score(stats, strategy, fold, y_true, y_pred)
                record[f"{strategy}_rmse"] = finite_float(rmse_from_arrays(y_true, y_pred))
            if config_get(config, "runtime.save_oof_predictions", False) and variant_name in oof_variant_names:
                add_oof_rows(
                    oof_records,
                    variant=variant_name,
                    variable=variant["variable"],
                    fold=fold,
                    well_id=well_id,
                    frame=frame,
                    y_true=y_true,
                    y_pred=predictions[variant_drift],
                )
            well_records.append(record)

    strategies = {
        strategy: summarize_strategy(strategy_stats)
        for strategy, strategy_stats in stats.items()
    }
    for strategy, strategy_stats in stats.items():
        for fold in range(n_splits):
            fold_records.append(
                {
                    "variant": variant_name,
                    "variable": variant["variable"],
                    "strategy": strategy,
                    "fold": fold,
                    "rmse": finite_float(
                        rmse_from_sums(
                            strategy_stats["fold_sse"][fold],
                            int(strategy_stats["fold_n"][fold]),
                        )
                    ),
                    "rows": int(strategy_stats["fold_n"][fold]),
                }
            )

    primary_metrics = strategies[variant_primary]
    ablation_record = {
        "variant": variant_name,
        "variable": variant["variable"],
        "description": variant["description"],
        "primary_strategy": variant_primary,
        "feature_set": config_get(variant_config, "model.feature_set", "all"),
        "n_features": len(feature_columns),
        "estimator": estimator,
        "max_rows_per_fold": max_rows_per_fold,
        "max_rows_final": max_rows_final,
        "max_rows_per_well": max_rows_per_well,
        "residual_shrink": config_get(variant_config, "model.params.residual_shrink", None),
        "cv": primary_metrics["oof_rmse"],
        "cv_mean_fold_rmse": primary_metrics["mean_fold_rmse"],
        "rows": primary_metrics["rows"],
        "delta_vs_exp002_cv": delta_vs_baseline(primary_metrics["oof_rmse"], baseline_cv),
        "delta_vs_exp003_cv": delta_vs_baseline(primary_metrics["oof_rmse"], cv_anchor_cv),
    }
    ablation_records.append(ablation_record)
    variant_summaries[variant_name] = {
        **ablation_record,
        "overrides": variant["overrides"],
        "feature_columns": feature_columns,
        "strategies": strategies,
    }
    print(
        f"  {variant_name} {variant_primary} CV RMSE:",
        primary_metrics["oof_rmse"],
        "delta_vs_exp002:",
        ablation_record["delta_vs_exp002_cv"],
    )

write_csv_artifacts(paths, ablation_records, well_records, fold_records, model_records, drift_name)
postprocess_records, selected_postprocess = evaluate_postprocess_candidates(oof_records, config, paths)

ablation_table = pd.DataFrame(ablation_records)
if not ablation_table.empty:
    print(ablation_table.sort_values("cv", na_position="last").to_string(index=False))
if postprocess_records:
    postprocess_table = pd.DataFrame(postprocess_records).sort_values("rmse", na_position="last")
    print(postprocess_table[["candidate", "method", "rmse", "delta_vs_raw", "source"]].to_string(index=False))
    print("Best postprocess:", selected_postprocess)


## 5. Metrics And Artifacts


In [ ]:
selected_variant = str(config_get(config, "ablation.selected_variant", variants[0]["name"]))
if selected_variant not in variant_summaries:
    print(f"Selected variant {selected_variant} was not run; using {variants[0]['name']} for top-level metrics")
    selected_variant = variants[0]["name"]

selected_summary = variant_summaries[selected_variant]
primary_metrics = selected_summary["strategies"][selected_summary["primary_strategy"]]
completed_records = [record for record in ablation_records if record["cv"] is not None]
best_record = min(completed_records, key=lambda record: record["cv"]) if completed_records else None

metrics = {
    "experiment": EXPERIMENT_NAME,
    "status": "debug_completed" if DEBUG else "completed",
    "created_at": config.get("experiment", {}).get("created_at"),
    "updated_at": datetime.now(UTC).isoformat(),
    "debug": DEBUG,
    "cv": primary_metrics["oof_rmse"],
    "cv_mean_fold_rmse": primary_metrics["mean_fold_rmse"],
    "public_lb": None,
    "selected_postprocess": selected_postprocess,
    "postprocess_records": postprocess_records,
    "private_lb": None,
    "metric": config.get("validation", {}).get("metric"),
    "seed": config.get("validation", {}).get("seed"),
    "primary_strategy": selected_summary["primary_strategy"],
    "selected_variant": selected_variant,
    "best_variant_by_cv": best_record["variant"] if best_record else None,
    "baseline_experiment": config_get(config, "ablation.baseline_experiment", None),
    "baseline_cv": config_get(config, "ablation.baseline_cv", None),
    "cv_anchor_experiment": config_get(config, "ablation.cv_anchor_experiment", None),
    "cv_anchor_cv": config_get(config, "ablation.cv_anchor_cv", None),
    "n_folds": n_splits,
    "n_wells": len(files),
    "strategies": selected_summary["strategies"],
    "ablation": {
        "records": ablation_records,
        "variants": variant_summaries,
    },
    "model_training": model_records,
    "key_idea": config.get("experiment", {}).get("description"),
    "notes": (
        "Postprocess and small model-diversity OOF comparison on top of exp012 LightGBM no-GR. "
        "Top-level cv tracks ablation.selected_variant; compare artifacts/postprocess_metrics.csv "
        "and artifacts/postprocess_distance_bucket_summary.csv before updating selected postprocess."
    ),
}
paths.metrics_path.write_text(json.dumps(metrics, indent=2) + "\n")

for record in ablation_records:
    print(f"{record['variant']} CV RMSE:", record["cv"])
print("Selected variant:", selected_variant)
print("Selected CV RMSE:", metrics["cv"])
print("Best variant by CV:", metrics["best_variant_by_cv"])
print("Selected postprocess candidate:", selected_postprocess)
print("Metrics written:", paths.metrics_path)
